In [17]:
import torch
import numpy as np

# Set random seed for reproducibility
torch.manual_seed(42)

# Generate random logprobs tensor
batch_size = 32
seq_length = 2048
vocab_size = 50000
logits = torch.randn(size=(batch_size, seq_length, vocab_size), requires_grad=True)

# Function to compute gradient in different dtypes
def compute_grad(logits, dtype):
    # Convert to specified dtype
    logits_dtype = logits.to(dtype).detach()
    logits_dtype.requires_grad_(True)
    
    # Sum over sequence length
    logprobs = torch.log_softmax(logits_dtype, dim=-1)
    sequence_logprobs = logprobs.amax(dim=-1)
    loss = sequence_logprobs.sum(dim=1).mean()
    
    # Compute gradient
    loss.backward()
    return logits_dtype.grad.to(torch.float32)  # Convert back to fp32 for comparison

# Compute gradients in fp32 and bf16
grad_fp32 = compute_grad(logits.clone(), torch.float32)
grad_bf16 = compute_grad(logits.clone(), torch.bfloat16)

# Calculate error metrics
abs_error = torch.abs(grad_fp32 - grad_bf16)
rel_error = abs_error / (torch.abs(grad_fp32) + 1e-11)  # Add epsilon to avoid division by zero

print(f"Average absolute error: {abs_error.mean().item():.6f}")
print(f"Average relative error: {rel_error.mean().item():.6f}")
print(f"Max absolute error: {abs_error.max().item():.6f}")
print(f"Max relative error: {rel_error.max().item():.6f}")


Average absolute error: 0.000000
Average relative error: 0.018117
Max absolute error: 0.023451
Max relative error: 1053.836914
